# Shared-core utilities validation (archival)

## Deterministic shared-core validation notebook

This notebook is **canonical** (see `notebook_plan.json`). It does **not** re-implement engine formulas. Instead it calls the same helper functions `evaluate_case` uses internally—`evaluate_gates`, `compute_compensatory_score`, `compute_abstention_rate`, `evaluate_batch`, and `hash_output`—so intermediate quantities can be audited independently of a full case verdict.


## Output contract

Exactly one artefact, per `trace_map.json`:

- `outputs/tables/utilities_validation.csv` — named checks and numeric or string values in deterministic order.

Changing row order, labels, or rounding would change the pinned SHA-256 and fail validation.


## Why these checks matter

- **Gates:** confirm the non-compensatory layer matches expectations on a fixed feature vector under the **moderate** profile.
- **Compensatory score:** exercises the canonical weighted sum (with bias inverted) on the same vector.
- **Abstention helper** (`compute_abstention_rate`): SCM logistic used in `canonical_full_mode`; here it is evaluated at 0.5 and at the case's uncertainty calibration to show the mapping is live code, not a stub.
- **Batch hash:** `evaluate_batch` on a one-case fixture proves batch serialisation and `hash_output` agree for a minimal replay-mode run.


In [1]:
from __future__ import annotations

import csv
import sys
from pathlib import Path


def repo_root() -> Path:
    start = Path.cwd().resolve()
    for p in (start, *start.parents):
        if (p / "engine" / "corrected_public_engine_v1_1.py").is_file():
            return p
    raise RuntimeError(
        "Repository root not found (expected engine/corrected_public_engine_v1_1.py)."
    )


ROOT = repo_root()
sys.path.insert(0, str(ROOT / "engine"))
import corrected_public_engine_v1_1 as eng

FEATURES = {
    "intrinsic_safety": 0.60,
    "evidence_strength": 0.58,
    "bias_harm_index": 0.42,
    "uncertainty_calibration": 0.55,
    "traceability_integrity": 0.56,
}

profile = eng.CANONICAL_THRESHOLD_PROFILES["moderate"]
gates = eng.evaluate_gates(FEATURES, profile)
comp_score = eng.compute_compensatory_score(FEATURES)
abstention_mid = eng.compute_abstention_rate(0.5)
abstention_uc = eng.compute_abstention_rate(FEATURES["uncertainty_calibration"])

fixture = {"case_id": "hash_fixture", "features": FEATURES}
batch = eng.evaluate_batch([fixture], profile_names=["moderate"], mode=eng.MODE_REPLAY)
batch_hash = eng.hash_output(batch)

rows = [
    ("evaluate_gates_all_pass", gates["all_gates_pass"]),
    ("binding_constraints_count", len(gates["binding_constraints"])),
    ("compensatory_score", round(comp_score, 6)),
    ("abstention_rate_at_uc_0.5", round(abstention_mid, 6)),
    ("abstention_rate_case_uc", round(abstention_uc, 6)),
    ("hash_batch_fixture_sha256", batch_hash),
]

out_path = ROOT / "outputs" / "tables" / "utilities_validation.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["check_name", "value"])
    for name, val in rows:
        w.writerow([name, val])

print("Utilities validation CSV written.")



Utilities validation CSV written.


## Reading `utilities_validation.csv`

Each row names a check that calls into `corrected_public_engine_v1_1.py`. The final row fingerprints the one-case batch object; auditors can compare it to an interactive session that calls the same API. This notebook performs **shared-core validation only**—not manuscript-specific claims.
